In [6]:
import os
import time
from itertools import product

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry


# =========================================================
# CONFIGURATION
# =========================================================

API_KEY = "Yc540b6fd190e428bbc5f0b0c862d824b"

BASE_URL = (
    "https://comtradeapi.un.org/data/v1/get/C/A/HS"
    if API_KEY
    else "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
)

YEARS = list(range(2018, 2025))

COUNTRIES = {
    "China": "156",
    "USA": "842",
    "Japan": "392",
    "Korea": "410",
    "Taiwan": "158",
    "Netherlands": "528",
    "Singapore": "702",
    "Germany": "276",
    "India": "356",
}

HS_CODES = [
    "854231",
    "854232",
    "854239",
]

FLOWS = {
    "Imports": "M",
    "Exports": "X",
}

OUTPUT_FILE = "v1_semiconductor_supply_chain.csv"


# =========================================================
# SESSION SETUP
# =========================================================

def make_session():

    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )

    session = requests.Session()

    session.mount(
        "https://",
        HTTPAdapter(max_retries=retry)
    )

    return session


# =========================================================
# DATA EXTRACTION
# =========================================================

def fetch_trade_data():

    session = make_session()

    all_data = []

    commodity_codes = ",".join(HS_CODES)

    country_pairs = list(product(COUNTRIES.items(), COUNTRIES.items()))

    total_requests = (
        len(country_pairs)
        * len(YEARS)
        * len(FLOWS)
    )

    progress = tqdm(
        total=total_requests,
        desc="Fetching Bilateral Semiconductor Trade Data"
    )

    for (
        reporter_name,
        reporter_code
    ), (
        partner_name,
        partner_code
    ) in country_pairs:

        # Skip same-country trade
        if reporter_code == partner_code:
            continue

        for year in YEARS:

            for flow_name, flow_code in FLOWS.items():

                params = {
                    "reporterCode": reporter_code,
                    "partnerCode": partner_code,
                    "flowCode": flow_code,
                    "cmdCode": commodity_codes,
                    "period": str(year),
                    "format": "JSON",
                    "maxRecords": 50000,
                    "includeDesc": "true",
                    "breakdownMode": "classic",
                }

                headers = {}

                if API_KEY:
                    headers[
                        "Ocp-Apim-Subscription-Key"
                    ] = API_KEY

                try:

                    response = session.get(
                        BASE_URL,
                        params=params,
                        headers=headers,
                        timeout=30,
                        verify=False
                    )

                    response.raise_for_status()

                    payload = response.json()

                except requests.RequestException as exc:

                    progress.write(
                        f"FAILED: "
                        f"{reporter_name} -> {partner_name} "
                        f"| {year} | {flow_name}"
                    )

                    progress.write(str(exc))

                    progress.update(1)

                    continue

                records = payload.get("data", [])

                if not records:

                    progress.write(
                        f"No data: "
                        f"{reporter_name} -> {partner_name} "
                        f"| {year} | {flow_name}"
                    )

                for row in records:

                    all_data.append({

                        "year":
                            row.get("period")
                            or row.get("refYear"),

                        "reporter":
                            row.get("reporterDesc")
                            or reporter_name,

                        "reporter_code":
                            row.get("reporterCode"),

                        "partner":
                            row.get("partnerDesc")
                            or partner_name,

                        "partner_code":
                            row.get("partnerCode"),

                        "flow":
                            row.get("flowDesc")
                            or flow_name,

                        "flow_code":
                            row.get("flowCode"),

                        "hs_code":
                            row.get("cmdCode"),

                        "commodity":
                            row.get("cmdDesc"),

                        "trade_value_usd":
                            row.get("primaryValue"),

                        "net_weight_kg":
                            row.get("netWgt"),
                    })

                progress.update(1)

                time.sleep(1)

    progress.close()

    return pd.DataFrame(all_data)


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    df = fetch_trade_data()

    print("\nDataset Shape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nSample Data:")
    print(df.head())

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(f"\nSaved to {OUTPUT_FILE}")

Fetching Bilateral Semiconductor Trade Data:   0%|          | 2/1134 [00:02<16:26,  1.15it/s]

FAILED: China -> USA | 2018 | Imports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=156&partnerCode=842&flowCode=M&cmdCode=854231%2C854232%2C854239&period=2018&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic
FAILED: China -> USA | 2018 | Exports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=156&partnerCode=842&flowCode=X&cmdCode=854231%2C854232%2C854239&period=2018&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic


Fetching Bilateral Semiconductor Trade Data:   0%|          | 4/1134 [00:02<07:43,  2.44it/s]

FAILED: China -> USA | 2019 | Imports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=156&partnerCode=842&flowCode=M&cmdCode=854231%2C854232%2C854239&period=2019&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic
FAILED: China -> USA | 2019 | Exports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=156&partnerCode=842&flowCode=X&cmdCode=854231%2C854232%2C854239&period=2019&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic


Fetching Bilateral Semiconductor Trade Data:   0%|          | 5/1134 [00:02<06:13,  3.03it/s]

...

FAILED: India -> Germany | 2022 | Exports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=356&partnerCode=276&flowCode=X&cmdCode=854231%2C854232%2C854239&period=2022&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic
FAILED: India -> Germany | 2023 | Imports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=356&partnerCode=276&flowCode=M&cmdCode=854231%2C854232%2C854239&period=2023&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic


Fetching Bilateral Semiconductor Trade Data:  89%|████████▉ | 1007/1134 [03:25<00:23,  5.41it/s]

FAILED: India -> Germany | 2023 | Exports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=356&partnerCode=276&flowCode=X&cmdCode=854231%2C854232%2C854239&period=2023&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic
FAILED: India -> Germany | 2024 | Imports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=356&partnerCode=276&flowCode=M&cmdCode=854231%2C854232%2C854239&period=2024&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic


Fetching Bilateral Semiconductor Trade Data:  89%|████████▉ | 1008/1134 [03:25<00:25,  4.90it/s]


FAILED: India -> Germany | 2024 | Exports
401 Client Error: Access Denied for url: https://comtradeapi.un.org/data/v1/get/C/A/HS?reporterCode=356&partnerCode=276&flowCode=X&cmdCode=854231%2C854232%2C854239&period=2024&format=JSON&maxRecords=50000&includeDesc=true&breakdownMode=classic

Dataset Shape:
(0, 0)

Columns:
[]

Sample Data:
Empty DataFrame
Columns: []
Index: []

Saved to v1_semiconductor_supply_chain.csv


In [ ]:
Yc540b6fd190e428bbc5f0b0c862d824b

In [7]:
import os
import time
from itertools import product

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry


# =========================================================
# CONFIGURATION
# =========================================================

API_KEY = os.getenv("COMTRADE_API_KEY", "").strip()

BASE_URL = (
    "https://comtradeapi.un.org/data/v1/get/C/A/HS"
    if API_KEY
    else "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
)

YEARS = list(range(2018, 2025))

COUNTRIES = {
    "China": "156",
    "USA": "842",
    "Japan": "392",
    "Korea": "410",
    "Taiwan": "158",
    "Netherlands": "528",
    "Singapore": "702",
    "Germany": "276",
    "India": "356",
}

HS_CODES = [
    "854231",
    "854232",
    "854239",
]

FLOWS = {
    "Imports": "M",
    "Exports": "X",
}

OUTPUT_FILE = "2ndTRY_semiconductor_supply_chain.csv"


# =========================================================
# SESSION SETUP
# =========================================================

def make_session():

    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )

    session = requests.Session()

    session.mount(
        "https://",
        HTTPAdapter(max_retries=retry)
    )

    return session


# =========================================================
# DATA EXTRACTION
# =========================================================

def fetch_trade_data():

    session = make_session()

    all_data = []

    commodity_codes = ",".join(HS_CODES)

    country_pairs = list(product(COUNTRIES.items(), COUNTRIES.items()))

    total_requests = (
        len(country_pairs)
        * len(YEARS)
        * len(FLOWS)
    )

    progress = tqdm(
        total=total_requests,
        desc="Fetching Bilateral Semiconductor Trade Data"
    )

    for (
        reporter_name,
        reporter_code
    ), (
        partner_name,
        partner_code
    ) in country_pairs:

        # Skip same-country trade
        if reporter_code == partner_code:
            continue

        for year in YEARS:

            for flow_name, flow_code in FLOWS.items():

                params = {
                    "reporterCode": reporter_code,
                    "partnerCode": partner_code,
                    "flowCode": flow_code,
                    "cmdCode": commodity_codes,
                    "period": str(year),
                    "format": "JSON",
                    "maxRecords": 50000,
                    "includeDesc": "true",
                    "breakdownMode": "classic",
                }

                headers = {}

                if API_KEY:
                    headers[
                        "Ocp-Apim-Subscription-Key"
                    ] = API_KEY

                try:

                    response = session.get(
                        BASE_URL,
                        params=params,
                        headers=headers,
                        timeout=30,
                        verify=False
                    )

                    response.raise_for_status()

                    payload = response.json()

                except requests.RequestException as exc:

                    progress.write(
                        f"FAILED: "
                        f"{reporter_name} -> {partner_name} "
                        f"| {year} | {flow_name}"
                    )

                    progress.write(str(exc))

                    progress.update(1)

                    continue

                records = payload.get("data", [])

                if not records:

                    progress.write(
                        f"No data: "
                        f"{reporter_name} -> {partner_name} "
                        f"| {year} | {flow_name}"
                    )

                for row in records:

                    all_data.append({

                        "year":
                            row.get("period")
                            or row.get("refYear"),

                        "reporter":
                            row.get("reporterDesc")
                            or reporter_name,

                        "reporter_code":
                            row.get("reporterCode"),

                        "partner":
                            row.get("partnerDesc")
                            or partner_name,

                        "partner_code":
                            row.get("partnerCode"),

                        "flow":
                            row.get("flowDesc")
                            or flow_name,

                        "flow_code":
                            row.get("flowCode"),

                        "hs_code":
                            row.get("cmdCode"),

                        "commodity":
                            row.get("cmdDesc"),

                        "trade_value_usd":
                            row.get("primaryValue"),

                        "net_weight_kg":
                            row.get("netWgt"),
                    })

                progress.update(1)

                time.sleep(1)

    progress.close()

    return pd.DataFrame(all_data)


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    df = fetch_trade_data()

    print("\nDataset Shape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nSample Data:")
    print(df.head())

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(f"\nSaved to {OUTPUT_FILE}")

Fetching Bilateral Semiconductor Trade Data:   4%|▍         | 43/1134 [01:07<28:17,  1.56s/it]

No data: China -> Taiwan | 2018 | Imports


Fetching Bilateral Semiconductor Trade Data:   4%|▍         | 44/1134 [01:09<28:32,  1.57s/it]

No data: China -> Taiwan | 2018 | Exports


Fetching Bilateral Semiconductor Trade Data:   4%|▍         | 45/1134 [01:10<27:04,  1.49s/it]

...

No data: India -> Germany | 2024 | Imports


Fetching Bilateral Semiconductor Trade Data:  89%|████████▉ | 1008/1134 [26:13<02:49,  1.34s/it]

No data: India -> Germany | 2024 | Exports


Fetching Bilateral Semiconductor Trade Data:  89%|████████▉ | 1008/1134 [26:14<03:16,  1.56s/it]



Dataset Shape:
(1764, 11)

Columns:
['year', 'reporter', 'reporter_code', 'partner', 'partner_code', 'flow', 'flow_code', 'hs_code', 'commodity', 'trade_value_usd', 'net_weight_kg']

Sample Data:
   year reporter  reporter_code partner  partner_code    flow flow_code  \
0  2018    China            156     USA           842  Import         M   
1  2018    China            156     USA           842  Import         M   
2  2018    China            156     USA           842  Import         M   
3  2018    China            156     USA           842  Export         X   
4  2018    China            156     USA           842  Export         X   

  hs_code                                          commodity  trade_value_usd  \
0  854231  Electronic integrated circuits; processors and...     9.848380e+09   
1  854232           Electronic integrated circuits; memories     7.173663e+08   
2  854239  Electronic integrated circuits; n.e.c. in head...     9.955156e+08   
3  854231  Electronic integr